# Muon 权衡 notebook 的相关形式化定理

这个 notebook 从实验 notebook 中抽取若干结论，并把它们整理成清楚的数学证书。目的不是证明每条非线性训练轨迹都必须和图像一致，而是把实验所用局部模型的确定性推论单独拎出来。

## 哪些结论适合证明

notebook 中有两类陈述：

- **模型推出的陈述。** 一旦接受局部线性化或指数价值模型，这些结论就由数学推出，适合形式化证明。
- **经验有效性陈述。** 这些陈述说模型在某次训练轨迹上预测得不错，依赖初始化、步长、真实轨迹和 Jacobian 漂移，应该由实验验证，而不是写成无条件定理。

下面形式化的是第一类陈述。

## 定理 1：正的局部速率给出一阶下降

黑盒 Jacobian 小节使用一阶近似

$$
\|e_{t+1}\|^2
\approx
\|e_t\|^2
\left(1-2\rho_t\right),
\qquad
\rho_t
=
\frac{\eta}{\|A\|_F^2}
\frac{e_t^\top J_tJ_t^\top e_t}{e_t^\top e_t}.
$$

其代数核心很简单但重要：

$$
\rho_t>0
\quad\Longrightarrow\quad
1-2\rho_t<1.
$$

因此，任何正的 Jacobian 有效速率都会预测一阶 loss 下降。

## 定理 2：未来速率优势可以抵消前缀 loss 代价

对两个调度 $a$ 和 $b$，定义

$$
\text{prefixRatio}
=
\frac{L(b)}{L(a)}
$$

并令 `totalGain` 表示 $b$ 相对 $a$ 的累计未来收缩优势。归一化后的预测 loss 比值是

$$
\text{prefixRatio}\cdot \exp(-\text{totalGain}).
$$

如果

$$
\log(\text{prefixRatio}) < \text{totalGain},
$$

那么

$$
\text{prefixRatio}\cdot \exp(-\text{totalGain}) < 1.
$$

这就是实验 notebook 中“当前或前缀 loss 更差的调度仍可能胜出”的严格数学形式：只要未来收缩优势足够大，它就会被预测为更好。

## 定理 3：多步收益在 log 空间中相加

多步 Jacobian 预测会把每一步的 log-ratio 估计相加。如果预测收益可以分解为

$$
\text{totalGain}=g_1+g_2+g_3,
$$

那么同一个证书可用于累计收益：

$$
\log(\text{prefixRatio}) < g_1+g_2+g_3
\quad\Longrightarrow\quad
\text{prefixRatio}\cdot \exp(-(g_1+g_2+g_3)) < 1.
$$

这解释了为什么 notebook 比较累计 log 衰减，而不仅仅看一步 loss 下降。

## Lean4 形式化

下面的 Lean4 代码在实数上证明这些代数证书。网络相关部分被故意放在定理之外：网络负责给出测得的 loss 和 Jacobian 速率；定理证明这些数值一旦满足条件，会推出什么排序结论。

**Lean 状态。** 下方 Lean 代码采用 Lean 4 core 风格，并已在远端服务器用 Lean 4.32.0 通过 `lean <file>.lean` 检查。


```lean
/-
Lean 4.32 core-verified log-domain tradeoff certificates.

The real-valued exponential predictor compares candidate b with baseline a:

  predicted_ratio = prefix_ratio * exp (- total_gain).

Taking logarithms gives the equivalent log-domain condition:

  log(predicted_ratio) = prefix_penalty - total_gain.

Thus predicted_ratio < 1 is certified by

  prefix_penalty < total_gain.

This file formalizes the log-domain algebra.  The real-analysis facts about
log and exp are standard; using this log form avoids a heavy Mathlib cache
dependency while still machine-checking the decision rule used by the notebooks.
-/

def logPredictedRatio (prefixPenalty totalGain : Int) : Int :=
  prefixPenalty - totalGain

theorem log_tradeoff_certificate
    {prefixPenalty totalGain : Int}
    (h : prefixPenalty < totalGain) :
    logPredictedRatio prefixPenalty totalGain < 0 := by
  unfold logPredictedRatio
  exact Int.sub_neg_of_lt h

def accumulatedGain3 (g1 g2 g3 : Int) : Int :=
  g1 + g2 + g3

theorem accumulated_log_tradeoff_certificate
    {prefixPenalty g1 g2 g3 : Int}
    (h : prefixPenalty < accumulatedGain3 g1 g2 g3) :
    logPredictedRatio prefixPenalty (accumulatedGain3 g1 g2 g3) < 0 := by
  exact log_tradeoff_certificate h

def firstOrderLossRatio (rho : Int) : Int :=
  1 - 2 * rho

theorem positive_rate_improves_first_order_loss
    {rho : Int}
    (hrho : 0 < rho) :
    firstOrderLossRatio rho < 1 := by
  unfold firstOrderLossRatio
  omega

def pointwiseImproves {n : Nat} (penalty gain : Fin n -> Int) : Prop :=
  forall t : Fin n, logPredictedRatio (penalty t) (gain t) < 0

theorem adjustment_method_pointwise_improves
    {n : Nat}
    {penalty gain : Fin n -> Int}
    (hcert : forall t : Fin n, penalty t < gain t) :
    pointwiseImproves penalty gain := by
  intro t
  exact log_tradeoff_certificate (hcert t)

```

## 解读

这些定理是刻意克制的。它们不声称 Muon、谱形状调整或某个固定调度永远更好。它们证明的是 notebook 中使用的决策逻辑：

1. 正的 Jacobian 有效速率预测一阶下降。
2. 未来收缩优势可以在数学上抵消当前 loss 劣势。
3. 多步证据应该在 log 空间中累计。

这足以避免结论被简单视为画图巧合。图像检验的是假设在具体系统中是否足够准确；定理证明的是，在这些假设成立时，排序规则本身是数学上必然的。